# Kaggle Pipeline: DDPG Only

This notebook runs DDPG experiments in batches with 3 seeds per experiment.
Use `BATCH_INDEX = 0..3` to select which quarter of the 12-config grid to run.
It is resume-safe and GPU-compatible.

In [ ]:
BASE_DIR = "/kaggle/working"

!git clone https://github.com/adityagangwani30/TD3-Car-Game.git
%cd TD3-Car-Game

import os
import shutil
from pathlib import Path

LOGS_DIR = Path(BASE_DIR) / "logs"
MODELS_DIR = Path(BASE_DIR) / "models"
RESULTS_DIR = Path(BASE_DIR) / "results"

for d in [LOGS_DIR, MODELS_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Route repository outputs to Kaggle working directories.
repo_root = Path.cwd()
for name, target in [("logs", LOGS_DIR), ("models", MODELS_DIR), ("results", RESULTS_DIR)]:
    link_path = repo_root / name
    if link_path.exists() or link_path.is_symlink():
        if link_path.is_symlink():
            link_path.unlink()
        elif link_path.is_dir():
            shutil.rmtree(link_path)
        else:
            link_path.unlink()
    os.symlink(target, link_path)

print("BASE_DIR:", BASE_DIR)
print("Working directory:", os.getcwd())
print("logs ->", (repo_root / "logs").resolve())
print("models ->", (repo_root / "models").resolve())
print("results ->", (repo_root / "results").resolve())

In [ ]:
!pip install -r requirements.txt

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("device =", device)

In [ ]:
MAX_EPISODES = 2000
MAX_STEPS_PER_EPISODE = 300
SEEDS = [0, 42, 123]
BATCHES = [
    (0, 3),   # experiments 0–2
    (3, 3),   # 3–5
    (6, 3),   # 6–8
    (9, 3)    # 9–11
]
BATCH_INDEX = 0  # user can change (0–3)
algo = "ddpg"
start_idx, num_exp = BATCHES[BATCH_INDEX]
EXPERIMENT_TAGS = [f"R{r}_N{n}" for r in range(1, 5) for n in range(1, 4)]
batch_experiments = EXPERIMENT_TAGS[start_idx:start_idx + num_exp]
print(f"Notebook role: DDPG only | Batch {BATCH_INDEX} -> start={start_idx}, count={num_exp}")

In [ ]:
for experiment in batch_experiments:
    for seed in SEEDS:
        print(f"[{algo.upper()}][{experiment}][Seed {seed}] Running...")
        !python run_experiments.py \
            --algo {algo} \
            --max-episodes 2000 \
            --max-steps 300 \
            --start-index {start_idx} \
            --max-experiments {num_exp} \
            --seed {seed} \
            --headless \
            --resume

In [ ]:
!python plot_metrics.py --compare-algos

In [ ]:
import shutil

shutil.make_archive('/kaggle/working/final_results', 'zip', '/kaggle/working/results')
print("Saved: /kaggle/working/final_results.zip")